# SNOTEL Daily SWE/Snow-Depth Validation (OL vs DA)

This notebook mirrors the Discover daily-cat workflow used in
`projects/M21C_ls/notebooks/snow_daily_seasonal_ol_da_from_discover.ipynb`,
but validates model snow variables against SNOTEL observations:

- Model `SNODPLAND` (snow depth, `m`) vs SNOTEL `SNWD` (converted from inches to meters)
- Model `SNOMASLAND` (SWE, `kg m-2`) vs SNOTEL `WTEQ` (converted from inches to `kg m-2`)

It also maps stations to nearest GEOSldas tiles and records tile elevation from GEOSldas tilecoord (`elev`).


## Workflow

1. Load SNOTEL parquet and station metadata (`lat`, `lon`, `elevation`).
2. Convert SNOTEL snow units from inches to model-comparable units.
3. Map stations to nearest GEOSldas tiles for OL and DA (with distance threshold).
4. Read daily cat files for `SNODPLAND` and `SNOMASLAND`.
5. Save raw intermediate timeseries cache (`NetCDF`) before computing statistics.
6. Compute AMS-style metrics (Bias, RMSE, ubRMSE, NSE) for full period and by season.


In [ ]:
# -------------------------
# Imports + configuration
# -------------------------
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

SEASON_ORDER = ["DJF", "MAM", "JJA", "SON"]

# Your Discover experiment definitions
EXPERIMENTS = {
    "OL": {
        "exp_name": "LS_OLv8_M36",
        "run_root": Path("/discover/nobackup/projects/land_da/M21C_land_sweeper/LS_OLv8_M36_v2/LS_OLv8_M36"),
    },
    "DA": {
        "exp_name": "LS_DAv8_M36",
        "run_root": Path("/discover/nobackup/projects/land_da/M21C_land_sweeper/LS_DAv8_M36_v3/LS_DAv8_M36"),
    },
}
DOMAIN = "SMAP_EASEv2_M36_GLOBAL"

# Analysis date span (inclusive)
ANALYSIS_START = "2000-06-01"
ANALYSIS_END = "2024-06-01"

# Station->tile mapping controls
DISTANCE_METHOD = "haversine_km"  # "haversine_km" or "squared_degree"
MAX_DISTANCE_KM = 40.0
MAX_DISTANCE_DEG2 = 0.10

# SNOTEL elevation metadata units from parquet (set to "ft", "m", or "unknown")
SNOTEL_ELEVATION_UNITS = "ft"

# Raw cache controls
USE_RAW_TIMESERIES_CACHE = True
WRITE_RAW_TIMESERIES_CACHE = True
WRITE_COMBINED_LONG_PARQUET = False  # optional; can be large

# Locate repo root robustly
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "common/python/io/read_GEOSldas.py").exists():
    for parent in REPO_ROOT.parents:
        if (parent / "common/python/io/read_GEOSldas.py").exists():
            REPO_ROOT = parent
            break

if not (REPO_ROOT / "common/python/io/read_GEOSldas.py").exists():
    raise FileNotFoundError("Could not find common/python/io/read_GEOSldas.py from current working directory")

PROJECT_ROOT = REPO_ROOT / "projects" / "SNOTEL"
NOTEBOOK_OUTPUT_DIR = PROJECT_ROOT / "outputs_snotel_ol_da_validation"
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SNOTEL_PARQUET = PROJECT_ROOT / "output" / "all_stations_daily_wteq_snwd.parquet"

_cache_tag = f"{DOMAIN}_{pd.Timestamp(ANALYSIS_START).strftime('%Y%m%d')}_{pd.Timestamp(ANALYSIS_END).strftime('%Y%m%d')}"
RAW_TIMESERIES_NC = NOTEBOOK_OUTPUT_DIR / f"snotel_raw_timeseries_{_cache_tag}.nc"
COMBINED_LONG_PARQUET = NOTEBOOK_OUTPUT_DIR / f"snotel_obs_model_daily_joined_{_cache_tag}.parquet"

STATION_MAP_CSV = {
    k: NOTEBOOK_OUTPUT_DIR / f"station_tile_map_{k}_{_cache_tag}.csv"
    for k in EXPERIMENTS.keys()
}

STATION_METRICS_CSV = NOTEBOOK_OUTPUT_DIR / f"snotel_station_metrics_{_cache_tag}.csv"
STATION_METRICS_PARQUET = NOTEBOOK_OUTPUT_DIR / f"snotel_station_metrics_{_cache_tag}.parquet"
DOMAIN_METRICS_CSV = NOTEBOOK_OUTPUT_DIR / f"snotel_domain_metrics_{_cache_tag}.csv"
DOMAIN_METRICS_PARQUET = NOTEBOOK_OUTPUT_DIR / f"snotel_domain_metrics_{_cache_tag}.parquet"
ELEVATION_MATCH_CSV = NOTEBOOK_OUTPUT_DIR / f"snotel_station_tile_elevation_{_cache_tag}.csv"

# Repo-local import for tilecoord
sys.path.insert(0, str(REPO_ROOT / "common/python/io"))
from read_GEOSldas import read_tilecoord  # type: ignore

print(f"REPO_ROOT={REPO_ROOT}")
print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"SNOTEL_PARQUET={SNOTEL_PARQUET}")
print(f"DOMAIN={DOMAIN}")
print(f"ANALYSIS_START={ANALYSIS_START}")
print(f"ANALYSIS_END={ANALYSIS_END}")
print(f"DISTANCE_METHOD={DISTANCE_METHOD}")
print(f"MAX_DISTANCE_KM={MAX_DISTANCE_KM}")
print(f"MAX_DISTANCE_DEG2={MAX_DISTANCE_DEG2}")
print(f"SNOTEL_ELEVATION_UNITS={SNOTEL_ELEVATION_UNITS}")
print(f"RAW_TIMESERIES_NC={RAW_TIMESERIES_NC}")
for k, cfg in EXPERIMENTS.items():
    print(f"{k}: exp_name={cfg['exp_name']}, run_root={cfg['run_root']}")


In [ ]:
# -------------------------
# Helpers
# -------------------------
def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorized Haversine distance in kilometers; lat2/lon2 can be arrays."""
    r = 6371.0
    p1 = np.deg2rad(lat1)
    p2 = np.deg2rad(lat2)
    dphi = np.deg2rad(lat2 - lat1)
    dlambda = np.deg2rad(lon2 - lon1)
    a = np.sin(dphi / 2.0) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlambda / 2.0) ** 2
    return 2.0 * r * np.arcsin(np.sqrt(a))


def locate_tilecoord_file(run_root: Path, exp_name: str, domain: str, output_dir: Path) -> Path:
    """Find tilecoord file with local-output override first."""
    candidates = [
        output_dir / "tilecoord.bin",
        output_dir / f"{exp_name}.ldas_tilecoord.bin",
        run_root / "output" / domain / "rc_out" / f"{exp_name}.ldas_tilecoord.bin",
        run_root / exp_name / "output" / domain / "rc_out" / f"{exp_name}.ldas_tilecoord.bin",
        run_root / f"{exp_name}.ldas_tilecoord.bin",
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("Could not find tilecoord file. Checked: " + ", ".join([str(p) for p in candidates]))


def locate_daily_cat_file(run_root: Path, exp_name: str, domain: str, day: pd.Timestamp):
    y = f"Y{day.year:04d}"
    m = f"M{day.month:02d}"
    stamp = day.strftime("%Y%m%d")
    fname = f"{exp_name}.tavg24_1d_lnd_Nt.{stamp}_1200z.nc4"
    candidates = [
        run_root / "output" / domain / "cat" / "ens_avg" / y / m / fname,
        run_root / exp_name / "output" / domain / "cat" / "ens_avg" / y / m / fname,
        run_root / "cat" / "ens_avg" / y / m / fname,
    ]
    for p in candidates:
        if p.exists():
            return p
    return None


def _read_var_1d_tile(ds, name):
    x = ds[name]
    if "time" in x.dims:
        x = x.isel(time=0)
    return np.asarray(x.values, dtype=np.float32)


def read_daily_snow_for_tiles(nc_path: Path, tile_indices: np.ndarray):
    """Read SNODPLAND and SNOMASLAND for selected tile indices from one daily file."""
    with xr.open_dataset(nc_path, decode_times=False) as ds:
        snod = _read_var_1d_tile(ds, "SNODPLAND")
        snomas = _read_var_1d_tile(ds, "SNOMASLAND")

    snod_sel = np.asarray(snod[tile_indices], dtype=np.float32)
    snomas_sel = np.asarray(snomas[tile_indices], dtype=np.float32)

    # Fill and simple physical checks
    snod_sel[snod_sel > 1e14] = np.nan
    snomas_sel[snomas_sel > 1e14] = np.nan
    snod_sel[snod_sel < 0.0] = np.nan
    snomas_sel[snomas_sel < 0.0] = np.nan

    return snod_sel, snomas_sel


def map_stations_to_tiles(
    tile_lat,
    tile_lon,
    tile_elev,
    station_df,
    distance_method="haversine_km",
    max_distance_deg2=0.1,
    max_distance_km=40.0,
):
    """Nearest-neighbor station->tile mapping with optional distance threshold."""
    rows = []
    tile_lat = np.asarray(tile_lat, dtype=float)
    tile_lon = np.asarray(tile_lon, dtype=float)
    tile_elev = np.asarray(tile_elev, dtype=float)

    for _, r in station_df.iterrows():
        stn = str(r["station"])
        lat = float(r["station_lat"])
        lon = float(r["station_lon"])
        station_elev_raw = float(r["station_elev_raw"]) if np.isfinite(r["station_elev_raw"]) else np.nan
        station_elev_m = float(r["station_elev_m"]) if np.isfinite(r["station_elev_m"]) else np.nan

        if not (np.isfinite(lat) and np.isfinite(lon)):
            continue

        if distance_method == "squared_degree":
            d_metric = (tile_lat - lat) ** 2 + (tile_lon - lon) ** 2
            j = int(np.argmin(d_metric))
            metric_val = float(d_metric[j])
            if max_distance_deg2 is not None and metric_val > float(max_distance_deg2):
                continue
            d_km = float(haversine_km(lat, lon, np.array([tile_lat[j]]), np.array([tile_lon[j]]))[0])

        elif distance_method == "haversine_km":
            d = haversine_km(lat, lon, tile_lat, tile_lon)
            j = int(np.argmin(d))
            d_km = float(d[j])
            if max_distance_km is not None and d_km > float(max_distance_km):
                continue
            metric_val = d_km

        else:
            raise ValueError(f"Unsupported distance_method={distance_method}")

        this_tile_elev = float(tile_elev[j]) if np.isfinite(tile_elev[j]) else np.nan
        elev_diff_m = np.nan
        if np.isfinite(this_tile_elev) and np.isfinite(station_elev_m):
            elev_diff_m = this_tile_elev - station_elev_m

        rows.append(
            {
                "station": stn,
                "station_lat": lat,
                "station_lon": lon,
                "station_elev_raw": station_elev_raw,
                "station_elev_m": station_elev_m,
                "tile_index": j,
                "tile_lat": float(tile_lat[j]),
                "tile_lon": float(tile_lon[j]),
                "tile_elev_m": this_tile_elev,
                "elev_diff_m": elev_diff_m,
                "distance_km": d_km,
                "distance_metric": metric_val,
                "distance_method": distance_method,
            }
        )

    out_cols = [
        "station", "station_lat", "station_lon", "station_elev_raw", "station_elev_m",
        "tile_index", "tile_lat", "tile_lon", "tile_elev_m", "elev_diff_m",
        "distance_km", "distance_metric", "distance_method",
    ]
    if len(rows) == 0:
        return pd.DataFrame(columns=out_cols)
    return pd.DataFrame(rows)[out_cols].sort_values("distance_km").reset_index(drop=True)


def build_obs_matrix(obs_df: pd.DataFrame, model_days: pd.DatetimeIndex, stations: list[str], value_col: str) -> np.ndarray:
    piv = obs_df.pivot_table(index="date", columns="station", values=value_col, aggfunc="mean")
    piv = piv.reindex(index=model_days, columns=stations)
    return np.asarray(piv.to_numpy(dtype=np.float32), dtype=np.float32)


def season_name(ts: pd.Timestamp) -> str:
    m = int(ts.month)
    if m in (12, 1, 2):
        return "DJF"
    if m in (3, 4, 5):
        return "MAM"
    if m in (6, 7, 8):
        return "JJA"
    return "SON"


def snow_error_metrics_ams(obs, mod, *, ddof_mean: int = 0):
    """
    Compute AMS-consistent error metrics for station-based snow validation.

    Matches the formulas:
      Bias  = mean(M - O)
      RMSE  = sqrt( (1/N) * sum( (O - M)^2 ) )
      ubRMSE = sqrt( (1/N) * sum( [ (M - mean(M)) - (O - mean(O)) ]^2 ) )
      NSE   = 1 - ( sum( (O - M)^2 ) / sum( (O - mean(O))^2 ) )

    Parameters
    ----------
    obs, mod : array-like
        Observations O_i and model estimates M_i. Same length. Can contain NaNs.
    ddof_mean : int, default 0
        Included for API compatibility (population mean over N samples is used).

    Returns
    -------
    metrics : dict
        {
          "N": int,
          "bias": float,
          "rmse": float,
          "ubrmse": float,
          "nse": float,
        }
    """
    _ = ddof_mean  # kept for compatibility with requested function signature

    o = np.asarray(obs, dtype=float)
    m = np.asarray(mod, dtype=float)
    if o.shape != m.shape:
        raise ValueError(f"obs and mod must have the same shape; got {o.shape} vs {m.shape}")

    valid = np.isfinite(o) & np.isfinite(m)
    o = o[valid]
    m = m[valid]
    N = o.size

    if N == 0:
        return {"N": 0, "bias": np.nan, "rmse": np.nan, "ubrmse": np.nan, "nse": np.nan}

    o_bar = np.mean(o)
    m_bar = np.mean(m)

    e = m - o
    bias = np.mean(e)
    rmse = np.sqrt(np.mean(e ** 2))
    ubrmse = np.sqrt(np.mean(((m - m_bar) - (o - o_bar)) ** 2))

    denom = np.sum((o - o_bar) ** 2)
    if denom == 0.0:
        nse = np.nan
    else:
        nse = 1.0 - (np.sum((o - m) ** 2) / denom)

    return {
        "N": int(N),
        "bias": float(bias),
        "rmse": float(rmse),
        "ubrmse": float(ubrmse),
        "nse": float(nse),
    }


In [ ]:
# -------------------------
# Load + preprocess SNOTEL observations
# -------------------------
if not SNOTEL_PARQUET.exists():
    raise FileNotFoundError(f"SNOTEL parquet not found: {SNOTEL_PARQUET}")

obs = pd.read_parquet(SNOTEL_PARQUET).copy()

required_cols = ["date", "stationTriplet", "latitude", "longitude", "WTEQ", "SNWD"]
missing_cols = [c for c in required_cols if c not in obs.columns]
if missing_cols:
    raise RuntimeError(f"Missing required columns in parquet: {missing_cols}")

obs["date"] = pd.to_datetime(obs["date"], errors="coerce").dt.normalize()
obs = obs.dropna(subset=["date", "stationTriplet", "latitude", "longitude"]).copy()
obs["station"] = obs["stationTriplet"].astype(str)

# Restrict analysis window
obs = obs[(obs["date"] >= pd.Timestamp(ANALYSIS_START)) & (obs["date"] <= pd.Timestamp(ANALYSIS_END))].copy()

# Convert SNOTEL inches -> model comparable units
# SNWD inches -> meters
# WTEQ inches water equivalent -> kg m-2 (1 inch = 25.4 mm water = 25.4 kg m-2)
obs["obs_snwd_in"] = pd.to_numeric(obs["SNWD"], errors="coerce")
obs["obs_swe_in"] = pd.to_numeric(obs["WTEQ"], errors="coerce")
obs["obs_snwd_m"] = obs["obs_snwd_in"] * 0.0254
obs["obs_swe_kgm2"] = obs["obs_swe_in"] * 25.4

# Station elevation from parquet metadata (if present)
if "elevation" in obs.columns:
    obs["station_elev_raw"] = pd.to_numeric(obs["elevation"], errors="coerce")
else:
    obs["station_elev_raw"] = np.nan

if str(SNOTEL_ELEVATION_UNITS).lower() == "ft":
    obs["station_elev_m"] = obs["station_elev_raw"] * 0.3048
elif str(SNOTEL_ELEVATION_UNITS).lower() == "m":
    obs["station_elev_m"] = obs["station_elev_raw"]
else:
    obs["station_elev_m"] = np.nan

# Daily station aggregation (guard against accidental duplicates)
obs_daily = (
    obs.groupby(["station", "date"], as_index=False)
    .agg(
        {
            "name": "first" if "name" in obs.columns else "first",
            "latitude": "first",
            "longitude": "first",
            "station_elev_raw": "first",
            "station_elev_m": "first",
            "obs_snwd_in": "mean",
            "obs_swe_in": "mean",
            "obs_snwd_m": "mean",
            "obs_swe_kgm2": "mean",
        }
    )
)

station_cols = ["station", "name", "latitude", "longitude", "station_elev_raw", "station_elev_m"]
station_df = (
    obs_daily[station_cols]
    .drop_duplicates("station")
    .rename(columns={"latitude": "station_lat", "longitude": "station_lon"})
    .sort_values("station")
    .reset_index(drop=True)
)

# Ensure clean station coordinates
station_df = station_df[np.isfinite(station_df["station_lat"]) & np.isfinite(station_df["station_lon"])].copy()
obs_daily = obs_daily[obs_daily["station"].isin(set(station_df["station"]))].copy()

print(f"Obs rows (daily): {len(obs_daily)}")
print(f"Unique stations with obs: {obs_daily['station'].nunique()}")
print(f"Obs date range: {obs_daily['date'].min().date()} to {obs_daily['date'].max().date()}")
display(station_df.head(10))


In [ ]:
# -------------------------
# Build or load raw intermediate timeseries cache (obs + model)
# -------------------------
loaded_from_cache = False
raw_ds = None
model_days = pd.date_range(pd.Timestamp(ANALYSIS_START), pd.Timestamp(ANALYSIS_END), freq="D")


def _cache_validation_errors(ds: xr.Dataset):
    errs = []

    required_vars = [
        "obs_snwd_in",
        "obs_swe_in",
        "obs_snwd_m",
        "obs_swe_kgm2",
        "model_snwd_m",
        "model_swe_kgm2",
        "station_lat",
        "station_lon",
        "tile_index",
        "tile_elev_m",
    ]
    for v in required_vars:
        if v not in ds.data_vars:
            errs.append(f"missing variable: {v}")

    if str(ds.attrs.get("domain", "")) != str(DOMAIN):
        errs.append(f"domain mismatch: cache={ds.attrs.get('domain')} vs config={DOMAIN}")

    if str(ds.attrs.get("analysis_start", "")) != str(ANALYSIS_START):
        errs.append(f"analysis_start mismatch: cache={ds.attrs.get('analysis_start')} vs config={ANALYSIS_START}")

    if str(ds.attrs.get("analysis_end", "")) != str(ANALYSIS_END):
        errs.append(f"analysis_end mismatch: cache={ds.attrs.get('analysis_end')} vs config={ANALYSIS_END}")

    expected_exp = [str(k) for k in EXPERIMENTS.keys()]
    got_exp = [str(x) for x in ds.coords["exp"].values] if "exp" in ds.coords else []
    if got_exp != expected_exp:
        errs.append(f"exp coord mismatch: cache={got_exp} vs config={expected_exp}")

    if "time" not in ds.coords:
        errs.append("missing coord: time")
    else:
        got_time = pd.to_datetime(ds["time"].values)
        if len(got_time) != len(model_days):
            errs.append(f"time length mismatch: cache={len(got_time)} vs expected={len(model_days)}")
        elif len(got_time) > 0:
            if pd.Timestamp(got_time[0]).normalize() != model_days[0]:
                errs.append(f"time start mismatch: cache={pd.Timestamp(got_time[0]).date()} vs expected={model_days[0].date()}")
            if pd.Timestamp(got_time[-1]).normalize() != model_days[-1]:
                errs.append(f"time end mismatch: cache={pd.Timestamp(got_time[-1]).date()} vs expected={model_days[-1].date()}")

    if "station" not in ds.coords:
        errs.append("missing coord: station")
    else:
        if int(ds.sizes.get("station", 0)) == 0:
            errs.append("station dimension is empty")

    return errs


if USE_RAW_TIMESERIES_CACHE and RAW_TIMESERIES_NC.exists():
    print(f"Found raw cache: {RAW_TIMESERIES_NC}")
    tmp_ds = xr.open_dataset(RAW_TIMESERIES_NC)
    cache_errs = _cache_validation_errors(tmp_ds)

    if len(cache_errs) == 0:
        raw_ds = tmp_ds
        loaded_from_cache = True
        print("Using existing raw cache (configuration match).")
    else:
        print("Existing raw cache does not match current configuration; running extraction.")
        for msg in cache_errs:
            print(f"  - {msg}")
        try:
            tmp_ds.close()
        except Exception:
            pass

elif USE_RAW_TIMESERIES_CACHE:
    print(f"Raw cache not found: {RAW_TIMESERIES_NC}")
    print("Running extraction from daily cat files...")
else:
    print("USE_RAW_TIMESERIES_CACHE=False; running extraction from daily cat files...")


if not loaded_from_cache:
    print("Running model extraction from daily cat files...")

    site_maps = {}
    for exp_key, cfg in EXPERIMENTS.items():
        exp_name = cfg["exp_name"]
        run_root = Path(cfg["run_root"])

        tilecoord_path = locate_tilecoord_file(run_root, exp_name, DOMAIN, NOTEBOOK_OUTPUT_DIR)
        tc = read_tilecoord(str(tilecoord_path))

        tile_lat_arr = np.asarray(tc["com_lat"], dtype=float)
        tile_lon_arr = np.asarray(tc["com_lon"], dtype=float)
        tile_elev_arr = np.asarray(tc["elev"], dtype=float) if "elev" in tc else np.full(tile_lat_arr.shape, np.nan, dtype=float)

        site_map_df = map_stations_to_tiles(
            tile_lat_arr,
            tile_lon_arr,
            tile_elev_arr,
            station_df,
            distance_method=DISTANCE_METHOD,
            max_distance_deg2=MAX_DISTANCE_DEG2,
            max_distance_km=MAX_DISTANCE_KM,
        )

        if len(site_map_df) == 0:
            raise RuntimeError(f"No stations mapped to tiles for {exp_key}")

        site_maps[exp_key] = site_map_df
        site_map_df.to_csv(STATION_MAP_CSV[exp_key], index=False)
        print(f"{exp_key}: mapped {len(site_map_df)} stations -> {STATION_MAP_CSV[exp_key]}")

    # Use only stations mapped for all experiments
    common_stations = sorted(set.intersection(*[set(df["station"]) for df in site_maps.values()]))
    if len(common_stations) == 0:
        raise RuntimeError("No common mapped stations across OL and DA")

    station_common_df = (
        station_df[station_df["station"].isin(common_stations)]
        .sort_values("station")
        .reset_index(drop=True)
    )
    common_stations = station_common_df["station"].tolist()

    # Build observation matrices aligned to (time, station)
    obs_common = obs_daily[obs_daily["station"].isin(common_stations)].copy()
    obs_snwd_in = build_obs_matrix(obs_common, model_days, common_stations, "obs_snwd_in")
    obs_swe_in = build_obs_matrix(obs_common, model_days, common_stations, "obs_swe_in")
    obs_snwd_m = build_obs_matrix(obs_common, model_days, common_stations, "obs_snwd_m")
    obs_swe_kgm2 = build_obs_matrix(obs_common, model_days, common_stations, "obs_swe_kgm2")

    exp_keys = list(EXPERIMENTS.keys())
    n_exp = len(exp_keys)
    n_time = len(model_days)
    n_station = len(common_stations)

    model_snwd_m = np.full((n_exp, n_time, n_station), np.nan, dtype=np.float32)
    model_swe_kgm2 = np.full((n_exp, n_time, n_station), np.nan, dtype=np.float32)

    tile_index = np.full((n_exp, n_station), -1, dtype=np.int32)
    tile_lat = np.full((n_exp, n_station), np.nan, dtype=np.float32)
    tile_lon = np.full((n_exp, n_station), np.nan, dtype=np.float32)
    tile_elev_m = np.full((n_exp, n_station), np.nan, dtype=np.float32)
    distance_km = np.full((n_exp, n_station), np.nan, dtype=np.float32)
    elev_diff_m = np.full((n_exp, n_station), np.nan, dtype=np.float32)

    for ei, exp_key in enumerate(exp_keys):
        cfg = EXPERIMENTS[exp_key]
        exp_name = cfg["exp_name"]
        run_root = Path(cfg["run_root"])

        m = (
            site_maps[exp_key]
            .set_index("station")
            .loc[common_stations]
            .reset_index()
        )

        tile_idx = m["tile_index"].to_numpy(dtype=int)
        tile_index[ei, :] = tile_idx.astype(np.int32)
        tile_lat[ei, :] = m["tile_lat"].to_numpy(dtype=np.float32)
        tile_lon[ei, :] = m["tile_lon"].to_numpy(dtype=np.float32)
        tile_elev_m[ei, :] = m["tile_elev_m"].to_numpy(dtype=np.float32)
        distance_km[ei, :] = m["distance_km"].to_numpy(dtype=np.float32)
        elev_diff_m[ei, :] = m["elev_diff_m"].to_numpy(dtype=np.float32)

        n_found = 0
        for ti, day in enumerate(model_days):
            f = locate_daily_cat_file(run_root, exp_name, DOMAIN, day)
            if f is None:
                continue

            snod_vals, snomas_vals = read_daily_snow_for_tiles(f, tile_idx)
            model_snwd_m[ei, ti, :] = snod_vals
            model_swe_kgm2[ei, ti, :] = snomas_vals
            n_found += 1

            if (ti + 1) % 365 == 0 or (ti + 1) == n_time:
                print(f"  {exp_key}: checked {ti + 1}/{n_time} days, files found={n_found}")

    # Station metadata aligned to common station order
    station_lat_vals = station_common_df["station_lat"].to_numpy(dtype=np.float32)
    station_lon_vals = station_common_df["station_lon"].to_numpy(dtype=np.float32)
    station_elev_raw_vals = station_common_df["station_elev_raw"].to_numpy(dtype=np.float32)
    station_elev_m_vals = station_common_df["station_elev_m"].to_numpy(dtype=np.float32)

    raw_ds = xr.Dataset(
        data_vars={
            "obs_snwd_in": (("time", "station"), obs_snwd_in),
            "obs_swe_in": (("time", "station"), obs_swe_in),
            "obs_snwd_m": (("time", "station"), obs_snwd_m),
            "obs_swe_kgm2": (("time", "station"), obs_swe_kgm2),
            "model_snwd_m": (("exp", "time", "station"), model_snwd_m),
            "model_swe_kgm2": (("exp", "time", "station"), model_swe_kgm2),
            "station_lat": (("station",), station_lat_vals),
            "station_lon": (("station",), station_lon_vals),
            "station_elev_raw": (("station",), station_elev_raw_vals),
            "station_elev_m": (("station",), station_elev_m_vals),
            "tile_index": (("exp", "station"), tile_index),
            "tile_lat": (("exp", "station"), tile_lat),
            "tile_lon": (("exp", "station"), tile_lon),
            "tile_elev_m": (("exp", "station"), tile_elev_m),
            "distance_km": (("exp", "station"), distance_km),
            "elev_diff_m": (("exp", "station"), elev_diff_m),
        },
        coords={
            "exp": np.array(exp_keys, dtype=object),
            "time": pd.DatetimeIndex(model_days),
            "station": np.array(common_stations, dtype=object),
        },
    )

    raw_ds.attrs["domain"] = DOMAIN
    raw_ds.attrs["analysis_start"] = str(ANALYSIS_START)
    raw_ds.attrs["analysis_end"] = str(ANALYSIS_END)
    raw_ds.attrs["distance_method"] = str(DISTANCE_METHOD)
    raw_ds.attrs["max_distance_km"] = float(MAX_DISTANCE_KM)
    raw_ds.attrs["max_distance_deg2"] = float(MAX_DISTANCE_DEG2)
    raw_ds.attrs["snotel_elevation_units"] = str(SNOTEL_ELEVATION_UNITS)
    raw_ds.attrs["exp_keys"] = ",".join([str(k) for k in exp_keys])
    raw_ds.attrs["exp_names"] = ",".join([str(EXPERIMENTS[k]["exp_name"]) for k in exp_keys])

    if WRITE_RAW_TIMESERIES_CACHE:
        print(f"Writing raw cache: {RAW_TIMESERIES_NC}")
        raw_ds.to_netcdf(RAW_TIMESERIES_NC)
        print(f"Wrote raw cache: {RAW_TIMESERIES_NC}")

print(raw_ds)
print(f"Stations in raw dataset: {raw_ds.sizes['station']}")
print(f"Dates in raw dataset: {raw_ds.sizes['time']}")


In [ ]:
# -------------------------
# Optional: write combined long parquet (large)
# -------------------------
if WRITE_COMBINED_LONG_PARQUET:
    exp_vals = [str(x) for x in raw_ds["exp"].values]
    station_vals = [str(x) for x in raw_ds["station"].values]
    time_vals = pd.to_datetime(raw_ds["time"].values)

    obs_snwd_m = np.asarray(raw_ds["obs_snwd_m"].values, dtype=float)
    obs_swe_kgm2 = np.asarray(raw_ds["obs_swe_kgm2"].values, dtype=float)
    obs_snwd_in = np.asarray(raw_ds["obs_snwd_in"].values, dtype=float)
    obs_swe_in = np.asarray(raw_ds["obs_swe_in"].values, dtype=float)
    model_snwd_m = np.asarray(raw_ds["model_snwd_m"].values, dtype=float)
    model_swe_kgm2 = np.asarray(raw_ds["model_swe_kgm2"].values, dtype=float)

    recs = []
    for ei, exp_key in enumerate(exp_vals):
        tmp = pd.DataFrame(
            {
                "date": np.repeat(time_vals.values, len(station_vals)),
                "station": np.tile(np.array(station_vals, dtype=object), len(time_vals)),
                "experiment": exp_key,
                "obs_snwd_in": obs_snwd_in.reshape(-1),
                "obs_swe_in": obs_swe_in.reshape(-1),
                "obs_snwd_m": obs_snwd_m.reshape(-1),
                "obs_swe_kgm2": obs_swe_kgm2.reshape(-1),
                "model_snwd_m": model_snwd_m[ei, :, :].reshape(-1),
                "model_swe_kgm2": model_swe_kgm2[ei, :, :].reshape(-1),
            }
        )
        recs.append(tmp)

    combined_long_df = pd.concat(recs, ignore_index=True)
    combined_long_df.to_parquet(COMBINED_LONG_PARQUET, index=False)
    print(f"Wrote combined long parquet: {COMBINED_LONG_PARQUET}")
else:
    print("WRITE_COMBINED_LONG_PARQUET=False; skipping combined long parquet write")


In [ ]:
# -------------------------
# Compute station + domain metrics
# -------------------------
exp_vals = [str(x) for x in raw_ds["exp"].values]
station_vals = [str(x) for x in raw_ds["station"].values]
time_vals = pd.to_datetime(raw_ds["time"].values)
season_vals = np.array([season_name(pd.Timestamp(t)) for t in time_vals], dtype=object)

obs_snwd_m = np.asarray(raw_ds["obs_snwd_m"].values, dtype=float)
obs_swe_kgm2 = np.asarray(raw_ds["obs_swe_kgm2"].values, dtype=float)
model_snwd_m = np.asarray(raw_ds["model_snwd_m"].values, dtype=float)
model_swe_kgm2 = np.asarray(raw_ds["model_swe_kgm2"].values, dtype=float)

station_lat = np.asarray(raw_ds["station_lat"].values, dtype=float)
station_lon = np.asarray(raw_ds["station_lon"].values, dtype=float)
station_elev_raw = np.asarray(raw_ds["station_elev_raw"].values, dtype=float)
station_elev_m = np.asarray(raw_ds["station_elev_m"].values, dtype=float)

tile_index = np.asarray(raw_ds["tile_index"].values)
tile_lat = np.asarray(raw_ds["tile_lat"].values, dtype=float)
tile_lon = np.asarray(raw_ds["tile_lon"].values, dtype=float)
tile_elev_m = np.asarray(raw_ds["tile_elev_m"].values, dtype=float)
distance_km = np.asarray(raw_ds["distance_km"].values, dtype=float)
elev_diff_m = np.asarray(raw_ds["elev_diff_m"].values, dtype=float)

season_masks = {"ALL": np.ones(len(time_vals), dtype=bool)}
for s in SEASON_ORDER:
    season_masks[s] = season_vals == s

station_records = []

var_specs = [
    ("SNODPLAND", obs_snwd_m, model_snwd_m, "m", "obs_snwd_m", "model_snwd_m"),
    ("SNOMASLAND", obs_swe_kgm2, model_swe_kgm2, "kg m-2", "obs_swe_kgm2", "model_swe_kgm2"),
]

for ei, exp_key in enumerate(exp_vals):
    for var_name, obs_arr, model_arr_3d, units, obs_col, model_col in var_specs:
        model_arr = model_arr_3d[ei, :, :]

        for sj, stn in enumerate(station_vals):
            o_full = obs_arr[:, sj]
            m_full = model_arr[:, sj]

            for season_name_key, mask in season_masks.items():
                o = o_full[mask]
                m = m_full[mask]
                metrics = snow_error_metrics_ams(o, m)

                valid = np.isfinite(o) & np.isfinite(m)
                obs_mean = float(np.mean(o[valid])) if np.any(valid) else np.nan
                model_mean = float(np.mean(m[valid])) if np.any(valid) else np.nan

                station_records.append(
                    {
                        "station": stn,
                        "experiment": exp_key,
                        "variable": var_name,
                        "units": units,
                        "season": season_name_key,
                        "obs_column": obs_col,
                        "model_column": model_col,
                        "N": metrics["N"],
                        "bias": metrics["bias"],
                        "rmse": metrics["rmse"],
                        "ubrmse": metrics["ubrmse"],
                        "nse": metrics["nse"],
                        "obs_mean": obs_mean,
                        "model_mean": model_mean,
                        "station_lat": station_lat[sj],
                        "station_lon": station_lon[sj],
                        "station_elev_raw": station_elev_raw[sj],
                        "station_elev_m": station_elev_m[sj],
                        "tile_index": int(tile_index[ei, sj]) if np.isfinite(tile_index[ei, sj]) else -1,
                        "tile_lat": tile_lat[ei, sj],
                        "tile_lon": tile_lon[ei, sj],
                        "tile_elev_m": tile_elev_m[ei, sj],
                        "distance_km": distance_km[ei, sj],
                        "elev_diff_m": elev_diff_m[ei, sj],
                    }
                )

station_metrics_df = pd.DataFrame(station_records)

# Domain-wide metrics (all stations pooled)
domain_records = []
for ei, exp_key in enumerate(exp_vals):
    for var_name, obs_arr, model_arr_3d, units, obs_col, model_col in var_specs:
        model_arr = model_arr_3d[ei, :, :]

        for season_name_key, mask in season_masks.items():
            o = obs_arr[mask, :].reshape(-1)
            m = model_arr[mask, :].reshape(-1)
            metrics = snow_error_metrics_ams(o, m)

            valid = np.isfinite(o) & np.isfinite(m)
            obs_mean = float(np.mean(o[valid])) if np.any(valid) else np.nan
            model_mean = float(np.mean(m[valid])) if np.any(valid) else np.nan

            domain_records.append(
                {
                    "experiment": exp_key,
                    "variable": var_name,
                    "units": units,
                    "season": season_name_key,
                    "obs_column": obs_col,
                    "model_column": model_col,
                    "N": metrics["N"],
                    "bias": metrics["bias"],
                    "rmse": metrics["rmse"],
                    "ubrmse": metrics["ubrmse"],
                    "nse": metrics["nse"],
                    "obs_mean": obs_mean,
                    "model_mean": model_mean,
                }
            )

domain_metrics_df = pd.DataFrame(domain_records)

# Write outputs
station_metrics_df.to_csv(STATION_METRICS_CSV, index=False)
station_metrics_df.to_parquet(STATION_METRICS_PARQUET, index=False)
domain_metrics_df.to_csv(DOMAIN_METRICS_CSV, index=False)
domain_metrics_df.to_parquet(DOMAIN_METRICS_PARQUET, index=False)

# Elevation match table (one row per station+experiment)
elev_match_df = (
    station_metrics_df[station_metrics_df["season"] == "ALL"][
        [
            "station",
            "experiment",
            "station_lat",
            "station_lon",
            "station_elev_raw",
            "station_elev_m",
            "tile_index",
            "tile_lat",
            "tile_lon",
            "tile_elev_m",
            "distance_km",
            "elev_diff_m",
        ]
    ]
    .drop_duplicates(["station", "experiment"])
    .sort_values(["experiment", "distance_km", "station"])
)
elev_match_df.to_csv(ELEVATION_MATCH_CSV, index=False)

print(f"Wrote station metrics CSV: {STATION_METRICS_CSV}")
print(f"Wrote station metrics parquet: {STATION_METRICS_PARQUET}")
print(f"Wrote domain metrics CSV: {DOMAIN_METRICS_CSV}")
print(f"Wrote domain metrics parquet: {DOMAIN_METRICS_PARQUET}")
print(f"Wrote elevation table CSV: {ELEVATION_MATCH_CSV}")

print("\nStation metrics shape:", station_metrics_df.shape)
print("Domain metrics shape:", domain_metrics_df.shape)

display(station_metrics_df.head(20))
display(domain_metrics_df.sort_values(["variable", "experiment", "season"]).head(20))
display(elev_match_df.head(20))


## Notes

- `tile_elev_m` comes from GEOSldas tilecoord (`read_tilecoord(...)["elev"]`).
- `station_elev_raw` is from SNOTEL metadata in the parquet file.
- `station_elev_m` is only populated if `SNOTEL_ELEVATION_UNITS` is set to `"ft"` or `"m"`.
- `elev_diff_m = tile_elev_m - station_elev_m` is only meaningful when `station_elev_m` is valid.
